# Feature Engineering — Beginner to Advanced

## Missing values • Imbalanced data • SMOTE • Outliers • Categorical encoding

**The goal:** turn messy real-world data into clean, useful inputs for a machine-learning model.

This notebook uses short explanations, tiny examples, runnable code, checkpoints, a final cheat sheet, and revision questions.

> **Simple mental picture:** raw data is groceries. Feature engineering is washing, cutting, and preparing them. The model is the cook. Even a genius cook cannot rescue rotten ingredients.

### Topics

1. What feature engineering is
2. Handling missing values
3. Handling imbalanced datasets
4. SMOTE
5. Detecting and treating outliers
6. One-hot (nominal) encoding
7. Label and ordinal encoding
8. Target-guided ordinal encoding
9. A safe end-to-end preprocessing workflow
10. Cheat sheet + revision questions and answers


## 0. How to use this notebook

- Read the **plain-English idea** first.
- Run the code cell below it.
- Predict the result before looking at the output.
- Read each **Advanced but important** box. Those boxes prevent very expensive mistakes.

### Words used throughout

| Word | Easy meaning |
|---|---|
| Row / observation / data point | One example, such as one customer |
| Column / variable / feature | One fact about every example, such as age |
| Target | What we want the model to predict |
| Training set | Data used to teach the model |
| Validation/test set | Unseen data used to judge the model |
| Transform | Change data into another form |
| Fit | Learn something from data, such as a mean or category list |
| Data leakage | Secretly letting training use information it should not know |


In [ ]:
# Beginner-friendly guide:
# These imports give us tools for number lists, tables, pictures, and machine-learning preparation.
# The fixed random seed makes our pretend random examples repeat in the same way each time.
# Core imports used in the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report
)
from sklearn.utils import resample

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
print("Setup complete.")


# 1. Feature engineering: the big picture

Most real datasets are messy:

- cells are empty;
- one class is much rarer than another;
- a few values are extremely large or small;
- useful categories are written as words;
- columns use different scales;
- some variables contain noise or leaked future information.

**Feature engineering** means preparing, transforming, selecting, or creating features so a model can learn the real pattern more easily.

### A safe workflow

1. Understand the problem and target.
2. Split the data into training and test sets.
3. Learn preprocessing rules from the **training set only**.
4. Apply the learned rules to validation/test data.
5. Train and evaluate with suitable metrics.

> The test set is the final exam. Do not show the answer sheet to the model during study time.


# 2. Handling missing values

## 2.1 What is a missing value?

A missing value means information was not stored for a variable.

Common reasons:

- a person skipped a survey question;
- a sensor failed;
- two databases did not match;
- a value was not applicable;
- someone made a data-entry mistake;
- the value was deliberately hidden.

Python usually shows missing values as `NaN`, `None`, or `pd.NA`.

## 2.2 The three missingness mechanisms

| Type | Full name | What controls the missingness? | Easy example |
|---|---|---|---|
| MCAR | Missing Completely At Random | Nothing in the observed or missing data | A random sensor packet is lost |
| MAR | Missing At Random | Other variables we can observe | Younger people skip a survey field more often |
| MNAR | Missing Not At Random | The missing value itself or an unmeasured cause | People with very high income avoid reporting income |

### Memory trick

- **MCAR:** missing for no systematic reason.
- **MAR:** another known column helps explain why it is missing.
- **MNAR:** the hidden value itself helps explain why it is hidden.

### Why the type matters

- Simple deletion is least dangerous under MCAR, but can still waste data.
- MAR can often be handled using information in other columns.
- MNAR is the hardest. A model cannot magically recover information that was systematically hidden. You may need domain knowledge, sensitivity analysis, or better data collection.


In [ ]:
# Beginner-friendly guide:
# This is a tiny practice passenger table; np.nan means that piece of information is missing.
# The last small table counts missing pieces so we can see which columns need attention.
# A small Titanic-like dataset created locally, so this notebook needs no internet.
passengers = pd.DataFrame({
    "age": [22, 38, np.nan, 35, np.nan, 54, 2, 27, np.nan, 44],
    "fare": [7.25, 71.28, 7.93, 53.10, 8.05, 51.86, 21.08, 11.13, 30.00, 13.00],
    "embarked": ["S", "C", "S", "S", np.nan, "S", "S", "Q", np.nan, "S"],
    "deck": [np.nan, "C", np.nan, "C", np.nan, "E", "G", np.nan, np.nan, np.nan],
    "survived": [0, 1, 1, 1, 0, 0, 1, 1, 0, 0],
})

print("Dataset shape:", passengers.shape)
display(passengers)

# True means the cell is missing. sum() counts True values.
missing_report = pd.DataFrame({
    "missing_count": passengers.isna().sum(),
    "missing_percent": passengers.isna().mean().mul(100).round(1),
})
display(missing_report)


## 2.3 Option A — delete missing data

### Delete rows

`df.dropna()` removes every row containing at least one missing value.

Use it only when:

- very few rows are missing;
- the missingness is reasonably close to MCAR;
- losing those rows will not remove an important group;
- enough data will remain.

### Delete columns

Dropping a column can make sense when:

- most values are missing;
- the column is not important;
- another column contains the same information;
- reliable imputation is impossible.

**Never use a blind rule such as “always drop above 50%.”** Importance and missingness mechanism matter too.


In [ ]:
# Beginner-friendly guide:
# We compare three choices: drop every incomplete row, drop every incomplete column, or remove just one chosen column.
# Looking at the shapes helps us see how much data each choice throws away.
rows_complete = passengers.dropna()
columns_complete = passengers.dropna(axis=1)

print("Original shape:            ", passengers.shape)
print("After dropping every row: ", rows_complete.shape)
print("After dropping any column:", columns_complete.shape)

# More controlled: remove only a mostly-empty column.
without_deck = passengers.drop(columns=["deck"])
print("After deliberately removing deck:", without_deck.shape)


## 2.4 Option B — impute (fill) missing values

### Mean imputation

Replace missing numbers with the column average.

- Works best when the data is roughly symmetric and has no strong outliers.
- Pulls missing observations toward the centre.
- Can shrink variance and weaken relationships between variables.

### Median imputation

Replace missing numbers with the middle value.

- Better for skewed data or data with outliers.
- More robust because one giant value barely moves the median.

### Mode imputation

Replace missing values with the most frequent value.

- Common for categorical columns.
- Can make the most common category look even more common.

### Random-sample imputation

For each empty cell, randomly copy an observed value from the same column.

- Better preserves the column's distribution than one fixed mean.
- Adds randomness, so use a fixed random seed.
- Still ignores relationships with other columns unless done within groups or with a model.


In [ ]:
# Beginner-friendly guide:
# We make a copy so the original table stays safe, then fill missing values in several simple ways.
# The random fill borrows real ages from known rows, and the fixed seed makes that choice repeatable.
filled = passengers.copy()

filled["age_mean"] = filled["age"].fillna(filled["age"].mean())
filled["age_median"] = filled["age"].fillna(filled["age"].median())
filled["embarked_mode"] = filled["embarked"].fillna(filled["embarked"].mode()[0])

# Random-sample imputation, reproducible because random_state is fixed.
observed_ages = filled["age"].dropna()
missing_mask = filled["age"].isna()
filled["age_random"] = filled["age"]
filled.loc[missing_mask, "age_random"] = observed_ages.sample(
    n=missing_mask.sum(), replace=True, random_state=RANDOM_STATE
).to_numpy()

display(filled[["age", "age_mean", "age_median", "age_random",
                "embarked", "embarked_mode"]])


## 2.5 Better practice with scikit-learn

`SimpleImputer` learns the fill value during `fit()` and uses it during `transform()`.

This matters because the test set must not influence the mean, median, or mode used for training.

### Missing-indicator feature

Sometimes “this value was missing” is itself useful information. `add_indicator=True` adds a 0/1 flag.

Example: if a medical test is ordered only for very sick patients, the absence or presence of the test can contain information.


In [ ]:
# Beginner-friendly guide:
# We split data first, then teach the imputer using training data only so the test data stays a fair surprise.
# add_indicator=True also adds a small flag that tells the model where information was missing.
X_train, X_test = train_test_split(
    passengers[["age", "fare"]], test_size=0.3, random_state=RANDOM_STATE
)

median_imputer = SimpleImputer(strategy="median", add_indicator=True)
X_train_imputed = median_imputer.fit_transform(X_train)  # learn from train
X_test_imputed = median_imputer.transform(X_test)        # reuse train rules

print("Learned median values:", median_imputer.statistics_)
print("Training output shape:", X_train_imputed.shape)
print("Test output shape:    ", X_test_imputed.shape)


## 2.6 Advanced missing-value methods

| Method | Idea | Useful when | Main warning |
|---|---|---|---|
| Group-wise imputation | Fill using a subgroup median/mode | Groups differ meaningfully | Small groups are unstable |
| KNN imputation | Use similar rows to estimate a value | Nearby cases are genuinely similar | Scale matters; can be slow |
| Iterative/model imputation | Predict each missing feature from other features | Relationships are strong | More assumptions and complexity |
| Constant value | Fill with `"Unknown"` or a fixed number | Missingness is a valid state | Fixed numbers can create fake meaning |
| Multiple imputation | Create several plausible completed datasets | Statistical inference and uncertainty matter | More work; combine results properly |

### Missing-value decision checklist

1. How much is missing in each column and row?
2. Is it MCAR, MAR, or possibly MNAR?
3. Is the variable important?
4. Is the variable numeric, nominal, or ordinal?
5. Is the distribution symmetric or skewed?
6. Will the method change the target relationship?
7. Was the imputer fitted only on training data?
8. Did cross-validation confirm the choice?


# 3. Handling imbalanced datasets

## 3.1 What does “imbalanced” mean?

In classification, one class has many more examples than another.

Example:

- 900 normal transactions
- 100 fraudulent transactions

The normal class is the **majority class**. Fraud is the **minority class**.

Imbalance is not automatically bad. It becomes a problem when the model ignores the rare class that we care about.

## 3.2 Why accuracy can lie

If 99 of 100 patients are healthy, a silly model that always says “healthy” gets 99% accuracy and finds zero sick patients.

Use these metrics too:

| Metric | Question it answers |
|---|---|
| Precision | Of the cases predicted positive, how many were truly positive? |
| Recall / sensitivity | Of all real positives, how many did we find? |
| F1 score | What is the balance between precision and recall? |
| Balanced accuracy | What is the average recall across classes? |
| Confusion matrix | Exactly which errors did the model make? |
| PR-AUC | How well does ranking work when the positive class is rare? |


In [ ]:
# Beginner-friendly guide:
# We build pretend data with many class-0 rows and only a few class-1 rows, like an unfairly balanced game.
# This lets us safely practise ways to handle imbalanced classes.
# Create a 90:10 imbalanced dataset like the lecture example.
n_majority, n_minority = 900, 100

majority = pd.DataFrame({
    "feature_1": np.random.normal(0, 1, n_majority),
    "feature_2": np.random.normal(0, 1, n_majority),
    "target": 0,
})
minority = pd.DataFrame({
    "feature_1": np.random.normal(2, 1, n_minority),
    "feature_2": np.random.normal(2, 1, n_minority),
    "target": 1,
})
imbalanced_df = pd.concat([majority, minority], ignore_index=True)

display(imbalanced_df["target"].value_counts().rename("count"))


## 3.3 Random oversampling (upsampling)

Random oversampling copies minority rows until the classes are more balanced.

**Good:** keeps all majority data; simple; often a useful baseline.

**Bad:** repeated rows add no new information and can encourage overfitting.

## 3.4 Random undersampling (downsampling)

Random undersampling removes majority rows.

**Good:** fast; useful when the majority class is enormous and repetitive.

**Bad:** throws away information and may remove rare but important majority patterns.

> Resample the **training set only**. Never balance the test set. The test set should represent the real world.


In [ ]:
# Beginner-friendly guide:
# Oversampling copies rare rows until both teams are the same size; undersampling keeps fewer common rows.
# We print the counts before and after so the balancing change is easy to see.
majority_rows = imbalanced_df[imbalanced_df["target"] == 0]
minority_rows = imbalanced_df[imbalanced_df["target"] == 1]

# Oversampling: copy minority rows WITH replacement until there are 900.
minority_up = resample(
    minority_rows,
    replace=True,
    n_samples=len(majority_rows),
    random_state=RANDOM_STATE,
)
upsampled_df = pd.concat([majority_rows, minority_up], ignore_index=True)

# Undersampling: choose majority rows WITHOUT replacement until there are 100.
majority_down = resample(
    majority_rows,
    replace=False,
    n_samples=len(minority_rows),
    random_state=RANDOM_STATE,
)
downsampled_df = pd.concat([majority_down, minority_rows], ignore_index=True)

print("Original:\n", imbalanced_df["target"].value_counts().sort_index())
print("\nAfter oversampling:\n", upsampled_df["target"].value_counts().sort_index())
print("\nAfter undersampling:\n", downsampled_df["target"].value_counts().sort_index())


## 3.5 Other strong options

- **Class weights:** punish mistakes on the minority class more heavily. Often try this first for linear models and trees.
- **Decision-threshold tuning:** predict positive at 0.30 instead of automatically using 0.50 when missing positives is costly.
- **Collect more minority data:** usually the cleanest solution if possible.
- **Special ensembles:** balanced random forests or boosting methods can help.
- **Anomaly detection:** useful when positives are extremely rare and different labels are limited.

### Do not force a perfect 50:50 balance automatically

The best ratio depends on the problem, model, metric, and cost of each error. Treat the sampling ratio as a hyperparameter.


In [ ]:
# Beginner-friendly guide:
# This lazy model always says class 0, so it looks accurate only because class 0 is very common.
# Balanced accuracy and recall reveal that it never finds the rare class-1 examples.
# Why accuracy can fool us
y_true = np.array([0] * 990 + [1] * 10)
y_lazy = np.zeros(1000, dtype=int)  # predicts only the majority class

print("Accuracy:          ", accuracy_score(y_true, y_lazy))
print("Balanced accuracy: ", balanced_accuracy_score(y_true, y_lazy))
print("Recall for class 1:", recall_score(y_true, y_lazy, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_true, y_lazy))


# 4. SMOTE — Synthetic Minority Over-sampling Technique

## 4.1 The five-year-old explanation

Random oversampling photocopies rare examples.

SMOTE creates new examples **between** nearby rare examples.

If two minority points are `A` and `B`, SMOTE chooses a random number `r` between 0 and 1:

\[
\text{new point} = A + r(B-A)
\]

- `r = 0` gives point A.
- `r = 1` gives point B.
- `r = 0.5` gives the midpoint.

## 4.2 Basic SMOTE algorithm

1. Choose a minority observation.
2. Find its nearest minority neighbours.
3. Randomly choose one neighbour.
4. Draw a synthetic point somewhere on the line between them.
5. Repeat until the requested class size is reached.

Unlike simple copying, this introduces new feature combinations and can spread the minority region.


In [ ]:
# Beginner-friendly guide:
# We make one new pretend point partway between two nearby minority points.
# This picture shows the simple geometric idea behind SMOTE, which creates helpful new minority examples.
# A tiny visual demonstration of interpolation between two minority points.
A = np.array([1.0, 2.0])
B = np.array([3.0, 4.0])
r = 0.35
synthetic = A + r * (B - A)

print("Point A:        ", A)
print("Point B:        ", B)
print("Synthetic point:", synthetic)

plt.figure(figsize=(6, 4))
plt.plot([A[0], B[0]], [A[1], B[1]], "--", color="gray", label="line between neighbours")
plt.scatter(*A, s=100, label="A")
plt.scatter(*B, s=100, label="B")
plt.scatter(*synthetic, s=120, marker="*", label="synthetic")
plt.legend()
plt.title("The core idea behind SMOTE")
plt.show()


In [ ]:
# Beginner-friendly guide:
# We try to use the optional SMOTE tool; if it is unavailable, the notebook gives a safe, clear message instead.
# SMOTE makes extra minority examples so a model can learn from a more balanced training set.
# Production implementation when imbalanced-learn is available.
# If it is not installed, the cell explains how to enable it and continues safely.
try:
    from imblearn.over_sampling import SMOTE

    X = imbalanced_df[["feature_1", "feature_2"]]
    y = imbalanced_df["target"]

    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X, y)

    print("Before:", y.value_counts().to_dict())
    print("After: ", pd.Series(y_smote).value_counts().to_dict())

    plt.figure(figsize=(6, 4))
    plt.scatter(X_smote.iloc[:, 0], X_smote.iloc[:, 1], c=y_smote, alpha=0.45, cmap="coolwarm")
    plt.title("After SMOTE")
    plt.xlabel("feature_1")
    plt.ylabel("feature_2")
    plt.show()
except ImportError:
    print("Optional package 'imbalanced-learn' is not installed.")
    print("Install it in your own environment with: pip install imbalanced-learn")


## 4.3 SMOTE warnings — extremely important

- Split first. Apply SMOTE only to the training fold.
- In cross-validation, SMOTE must run separately inside each training fold. Use an `imblearn` pipeline.
- Scale numeric features before neighbour-based sampling when their units differ greatly.
- Do not apply ordinary SMOTE directly to raw categorical codes. Use methods such as `SMOTENC` for mixed numeric/categorical data.
- SMOTE can create ambiguous points between overlapping classes.
- It can amplify noisy or incorrectly labelled minority examples.
- Time-series and grouped data need special splits; synthetic points must not break time or patient boundaries.
- Synthetic data is not new real-world evidence. It helps the model learn; it does not increase the true sample size for scientific inference.

### When SMOTE may be a bad choice

- only a handful of minority cases exist;
- minority examples are noisy;
- classes overlap heavily;
- the feature space is mostly categorical;
- distances are not meaningful;
- your model already handles class weights well.


# 5. Handling outliers

## 5.1 What is an outlier?

An outlier is a value far away from most other values.

An outlier may be:

- a typing error, such as age = 999;
- a measurement failure;
- a valid rare case, such as an extremely expensive house;
- evidence of fraud or disease—the exact thing we want to detect.

**Do not delete a value just because a formula calls it an outlier. Investigate first.**

## 5.2 Five-number summary

1. Minimum
2. First quartile, Q1 (25th percentile)
3. Median, Q2 (50th percentile)
4. Third quartile, Q3 (75th percentile)
5. Maximum

The interquartile range is:

\[
IQR = Q3 - Q1
\]

Common outlier fences:

\[
\text{Lower fence}=Q1-1.5(IQR)
\]

\[
\text{Upper fence}=Q3+1.5(IQR)
\]

Values beyond these fences are flagged as possible outliers.


In [ ]:
# Beginner-friendly guide:
# We find the middle half of the marks, then use its width (IQR) to mark unusually far-away values.
# The box plot turns those number rules into a quick picture.
marks = np.array([-200, -100, 32, 32, 45, 45, 54, 54, 56, 67, 67,
                  74, 75, 87, 89, 89, 90, 98, 99, 150, 170, 180])

minimum, q1, median, q3, maximum = np.quantile(marks, [0, 0.25, 0.5, 0.75, 1])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outliers = marks[(marks < lower_fence) | (marks > upper_fence)]

print({
    "minimum": minimum, "Q1": q1, "median": median,
    "Q3": q3, "maximum": maximum, "IQR": iqr,
    "lower_fence": lower_fence, "upper_fence": upper_fence,
})
print("Flagged values:", outliers)

plt.figure(figsize=(8, 2.5))
sns.boxplot(x=marks)
plt.title("Box plot: dots beyond the whiskers are flagged")
plt.show()


## 5.3 Reading a box plot

- The box runs from Q1 to Q3.
- The line inside the box is the median.
- The box width represents the middle 50% of values.
- Whiskers extend to the most extreme observations still inside the fences.
- Points beyond the whiskers are plotted separately.

### Important detail

The whisker ends are usually **not** the true minimum and maximum when outliers exist.

## 5.4 Other detection methods

| Method | Useful when | Caution |
|---|---|---|
| Domain rules | Known valid limits exist | Usually the best starting point |
| IQR rule | Skewed or non-normal numeric data | Can flag many valid points in long-tailed data |
| Z-score | Distribution is roughly normal | Mean and standard deviation are themselves sensitive to outliers |
| Modified Z-score / MAD | Robust detection is needed | Need a non-zero MAD |
| Scatter/residual plots | Relationship outliers matter | Requires visual or model-based judgement |
| Isolation Forest / LOF | Multivariate anomalies | More complex and parameter-sensitive |

An observation can look normal in every single column but still be unusual as a combination. That is a **multivariate outlier**.


## 5.5 What should we do with outliers?

1. **Correct** verified data-entry or unit errors.
2. **Keep** valid rare cases when they belong to the population.
3. **Remove** a point only with a defensible reason.
4. **Cap / winsorize** extreme values at chosen limits.
5. **Transform** a right-skewed positive feature using `log1p`.
6. **Use robust statistics**, such as median and IQR.
7. **Use robust models/scalers**, such as tree models or `RobustScaler`.
8. **Create an indicator**, such as `was_capped`, if extremeness may matter.
9. **Compare sensitivity:** report results with and without disputed observations.

> Fit thresholds on training data only. A test-set outlier must not change the training fences.


In [ ]:
# Beginner-friendly guide:
# We learn safe IQR limits from training values, then pull only extreme incomes back inside those limits.
# log1p is another gentle option that squeezes very large positive numbers closer together.
# IQR capping example: learn q1/q3 from training data, then reuse the limits.
train_income = pd.Series([35, 42, 47, 51, 55, 59, 63, 70, 500], name="income_k")

q1 = train_income.quantile(0.25)
q3 = train_income.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

income_capped = train_income.clip(lower=lower, upper=upper)
income_log = np.log1p(train_income)

display(pd.DataFrame({
    "original": train_income,
    "capped": income_capped,
    "log1p": income_log.round(3),
}))


# 6. One-hot encoding (nominal encoding)

## 6.1 Why encode categories?

Most machine-learning calculations need numbers. A value such as `"red"` must therefore be represented numerically.

## 6.2 Nominal categories

Nominal categories have **names but no natural ranking**.

Examples: colour, city, blood type, payment method.

For colours:

| Colour | blue | green | red |
|---|---:|---:|---:|
| blue | 1 | 0 | 0 |
| green | 0 | 1 | 0 |
| red | 0 | 0 | 1 |

This is **one-hot encoding (OHE)**. Each category gets its own 0/1 feature.

### Advantages

- does not invent a fake order;
- easy to understand;
- works well for low-cardinality nominal features.

### Disadvantages

- many categories create many columns;
- the matrix can become sparse;
- rare categories may encourage overfitting;
- unknown categories at prediction time need a plan.


In [ ]:
# Beginner-friendly guide:
# One-hot encoding gives every color its own yes-or-no column, so the model does not mistake colors for sizes.
# ignore lets a new color such as yellow pass through safely instead of causing an error.
colors = pd.DataFrame({"color": ["red", "blue", "green", "green", "red", "blue"]})

# handle_unknown='ignore' prevents a crash when a new category appears later.
# sparse_output=False returns a normal NumPy array for easy viewing.
try:
    color_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn versions
    color_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

encoded = color_encoder.fit_transform(colors[["color"]])
encoded_df = pd.DataFrame(encoded, columns=color_encoder.get_feature_names_out(), index=colors.index)

display(pd.concat([colors, encoded_df], axis=1))
print("New value 'yellow':", color_encoder.transform([["yellow"]]))


## 6.3 One-hot encoding details

### `fit` versus `transform`

- `fit`: learn the category list and column order.
- `transform`: encode rows using that fixed list.
- `fit_transform`: do both on training data.

Never fit a fresh encoder separately on the test set. Its columns may differ.

### Dummy-variable trap

With an intercept, all one-hot columns together contain redundant information because each row sums to 1. Linear and logistic regression may use `drop="first"` to reduce perfect multicollinearity. Regularisation also helps.

Tree models often do not require dropping a category.

### High cardinality

If a feature contains thousands of cities, products, or IDs, plain OHE may be huge. Options include:

- combine genuinely rare categories into `"Other"`;
- frequency/count encoding;
- hashing;
- carefully regularised target encoding;
- native categorical models;
- embeddings for large learned systems.

Do not encode meaningless IDs as if they were useful categories.


# 7. Label encoding and ordinal encoding

## 7.1 Label encoding

Label encoding gives every category a unique integer, such as:

- blue → 0
- green → 1
- red → 2

### Crucial distinction

`LabelEncoder` is mainly intended for the **target labels `y`**, such as `cat`, `dog`, and `bird`.

Using 0, 1, and 2 for a nominal input feature can create a fake relationship:

- the model may treat red as greater than green;
- it may treat the gap from blue to red as twice the gap from blue to green.

For nominal input features, OHE is usually safer.


In [ ]:
# Beginner-friendly guide:
# Label encoding turns target words into numbers so a classifier can learn from them.
# We also turn the numbers back into words to check that the translation is correct.
# Appropriate use: encode target labels.
animal_labels = np.array(["cat", "dog", "cat", "bird", "dog"])
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(animal_labels)

print("Classes in learned order:", label_encoder.classes_)
print("Encoded target:          ", y_encoded)
print("Back to words:           ", label_encoder.inverse_transform(y_encoded))


## 7.2 Ordinal encoding

Ordinal categories have a real order.

Examples:

- small < medium < large
- poor < fair < good < excellent
- high school < bachelor's < master's < doctorate

`OrdinalEncoder` lets us state the order explicitly.

### Important limitation

Codes 0, 1, 2 imply equal steps. Real-world steps may not be equal. The jump from `medium` to `large` may not equal the jump from `small` to `medium`.

For linear models, consider one-hot encoding or testing different representations when spacing is uncertain.


In [ ]:
# Beginner-friendly guide:
# Ordinal encoding is right for sizes because small, medium, and large have a real order.
# An unknown size gets -1 so the program can clearly recognise that it has not seen it before.
sizes = pd.DataFrame({"size": ["small", "medium", "large", "medium", "small"]})

size_encoder = OrdinalEncoder(
    categories=[["small", "medium", "large"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
)
sizes["size_encoded"] = size_encoder.fit_transform(sizes[["size"]])

display(sizes)
print("Unknown size 'extra-large':", size_encoder.transform([["extra-large"]]))


## 7.3 Quick encoder choice

| Situation | Good first choice |
|---|---|
| Target labels are words | `LabelEncoder` or a direct mapping |
| Input categories have no order | `OneHotEncoder` |
| Input categories have a trusted order | `OrdinalEncoder` |
| Thousands of input categories | Frequency, hashing, target encoding, or a native categorical model |
| Tree library supports categories natively | Use its documented native method |


# 8. Target-guided ordinal encoding

## 8.1 Simple idea

Replace each category with a number calculated from its target values.

Suppose the target is house price:

- London's mean price = 150
- New York's mean price = 190
- Tokyo's mean price = 250
- Paris's mean price = 310

The city name becomes its average target value.

This can be useful for high-cardinality features because it creates one column instead of thousands.

### Regression

Often use the mean or median target for each category.

### Binary classification

The category mean of a 0/1 target is the observed positive rate.


In [ ]:
# Beginner-friendly guide:
# Target encoding replaces each city with the average price seen for that city.
# This simple example is for learning; in real model work, learn the map from training data only to avoid leakage.
city_prices = pd.DataFrame({
    "city": ["New York", "London", "Paris", "Tokyo", "New York", "Paris"],
    "price": [200, 150, 300, 250, 180, 320],
})

mean_price_map = city_prices.groupby("city")["price"].mean().to_dict()
city_prices["city_target_mean"] = city_prices["city"].map(mean_price_map)

print("Learned map:", mean_price_map)
display(city_prices)


## 8.2 The giant trap: target leakage

The simple code above uses every row's target to encode that same row. A rare category can almost reveal its own answer.

Safe rules:

1. Split the data first.
2. Learn mappings only from the training data.
3. For validation/test rows, use the training mapping.
4. For training rows, use **out-of-fold (OOF)** encoding: each row is encoded using a mapping learned without its fold.
5. Use smoothing for rare categories.
6. Give unseen categories a fallback, often the global target mean.

## 8.3 Smoothing

A category with one row should not receive the same trust as a category with 10,000 rows.

One simple smoothed value is:

\[
\text{encoded} = \frac{n\times \text{category mean} + m\times \text{global mean}}{n+m}
\]

- `n`: number of rows in the category
- `m`: smoothing strength
- small `n` → value stays near global mean
- large `n` → value moves toward category mean


In [ ]:
# Beginner-friendly guide:
# These two helper functions learn a safer target-mean map and apply it to new data.
# Smoothing gently pulls tiny-category guesses toward the overall average, and unseen cities use that fallback.
def fit_smoothed_target_map(category, target, smoothing=10.0):
    # Learn a smoothed target-mean map from TRAINING data only.
    frame = pd.DataFrame({"category": category, "target": target})
    global_mean = frame["target"].mean()
    stats = frame.groupby("category")["target"].agg(["mean", "count"])
    stats["encoded"] = (
        stats["count"] * stats["mean"] + smoothing * global_mean
    ) / (stats["count"] + smoothing)
    return stats["encoded"].to_dict(), global_mean


def apply_target_map(category, mapping, fallback):
    # Apply the learned map; unseen categories receive the fallback.
    return pd.Series(category).map(mapping).fillna(fallback).to_numpy()


train = pd.DataFrame({
    "city": ["A", "A", "A", "B", "B", "C"],
    "bought": [1, 0, 1, 0, 0, 1],
})
test = pd.DataFrame({"city": ["A", "B", "D"]})  # D was unseen

mapping, global_mean = fit_smoothed_target_map(train["city"], train["bought"], smoothing=3)
test["city_encoded"] = apply_target_map(test["city"], mapping, global_mean)

print("Global target mean:", round(global_mean, 3))
print("Smoothed mapping:  ", {k: round(v, 3) for k, v in mapping.items()})
display(test)


## 8.4 When target encoding is useful

- a categorical column has many unique values;
- categories have enough repeated observations;
- the target relationship is reasonably stable;
- careful cross-validation and regularisation are used.

## 8.5 When it is risky

- tiny datasets;
- many categories appear once;
- future target information can leak backward in time;
- groups such as patients or users appear across folds;
- the relationship changes quickly;
- the encoded value may capture protected-group bias.

For time data, create mappings using the past only. For repeated people/items, split by group so one entity does not teach the encoding used on itself.


# 9. Safe end-to-end preprocessing

## 9.1 The central rule

Anything that **learns from data** must be fitted on the training data only:

- imputation values;
- scaling values;
- outlier thresholds;
- category lists;
- target-encoding maps;
- sampling or SMOTE;
- feature selection.

Pipelines make this rule easier to follow.

Below:

- numeric features receive median imputation and robust scaling;
- nominal features receive mode imputation and one-hot encoding;
- an ordinal feature receives mode imputation and explicit ordinal encoding;
- logistic regression receives class weights.


In [ ]:
# Beginner-friendly guide:
# This is a complete small workflow: split first, clean each type of feature, encode words, then train one model.
# The pipeline keeps every preparation step together so training and test data are handled in the same safe order.
# A compact, realistic mixed-type classification example.
model_df = pd.DataFrame({
    "age": [23, 35, np.nan, 52, 46, 31, 27, np.nan, 61, 40, 29, 55],
    "income": [45, 70, 52, 120, 95, 60, 49, 75, 140, 82, 55, 110],
    "city": ["Sydney", "Melbourne", "Sydney", "Perth", "Sydney", "Perth",
             "Melbourne", "Sydney", "Perth", "Melbourne", "Sydney", "Perth"],
    "plan": ["basic", "plus", "basic", "premium", "plus", "basic",
             "plus", "basic", "premium", "plus", "basic", "premium"],
    "churned": [0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1],
})

X = model_df.drop(columns="churned")
y = model_df["churned"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, stratify=y, random_state=RANDOM_STATE
)

numeric_features = ["age", "income"]
nominal_features = ["city"]
ordinal_features = ["plan"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", RobustScaler()),
])

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", ohe),
])

ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=[["basic", "plus", "premium"]],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    )),
])

preprocess = ColumnTransformer([
    ("numeric", numeric_pipe, numeric_features),
    ("nominal", nominal_pipe, nominal_features),
    ("ordinal", ordinal_pipe, ordinal_features),
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=RANDOM_STATE)),
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:         ", round(accuracy_score(y_test, pred), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, pred), 3))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))


## 9.2 Production checklist

### Before modelling

- [ ] Define the target and prediction time.
- [ ] Remove impossible future/leaked features.
- [ ] Choose random, stratified, grouped, or time-based splitting correctly.
- [ ] Inspect missingness, class balance, distributions, and categories.

### During preprocessing

- [ ] Fit every learned transformation on training data only.
- [ ] Preserve column names and document each transformation.
- [ ] Handle unseen categories.
- [ ] Keep sampling inside training folds.
- [ ] Use a pipeline where possible.

### During evaluation

- [ ] Compare against a simple baseline.
- [ ] Use metrics that match the real cost of errors.
- [ ] Evaluate important subgroups.
- [ ] Check calibration and decision thresholds when probabilities matter.
- [ ] Test sensitivity to imputation, outliers, and sampling choices.

### During deployment

- [ ] Save the fitted preprocessing and model together.
- [ ] Monitor missing rates, category drift, class balance, and feature ranges.
- [ ] Decide what happens when a required input is absent or invalid.


# 10. Master cheat sheet

## Missing values

| Situation | First option to test | Watch out for |
|---|---|---|
| Few random missing rows | Careful row deletion | Lost sample size or subgroup bias |
| Symmetric numeric feature | Mean | Outliers; reduced variance |
| Skewed/outlier-heavy numeric feature | Median | Still changes distribution |
| Categorical feature | Mode or `Unknown` | Inflating one category |
| Missingness itself is informative | Add indicator | Leakage from future processes |
| Complex relationships | KNN/model/multiple imputation | Complexity and assumptions |

**Mechanisms:** MCAR = unrelated; MAR = explained by observed data; MNAR = linked to hidden value/unobserved cause.

## Imbalanced classification

| Method | Main idea | Main cost |
|---|---|---|
| Oversampling | Copy minority rows | Overfitting |
| Undersampling | Remove majority rows | Information loss |
| SMOTE | Interpolate minority neighbours | Synthetic noise/overlap |
| Class weights | Penalise minority mistakes more | Requires tuning/evaluation |
| Threshold tuning | Change probability cutoff | Must use validation data |

**Metrics:** confusion matrix, precision, recall, F1, balanced accuracy, PR-AUC.

## Outliers

\[
IQR=Q3-Q1,\quad L=Q1-1.5IQR,\quad U=Q3+1.5IQR
\]

Investigate → correct errors → keep valid cases or justify capping/transformation/removal.

## Encoding

| Data type | Common encoding |
|---|---|
| Nominal input | One-hot |
| Ordered input | Ordinal |
| Text target labels | Label encoding |
| High-cardinality input | Regularised target/frequency/hashing/native method |

## The anti-leakage commandment

**Split first. Fit preprocessing on training data only. Transform validation/test using the learned rules.**


# 11. Revision questions and answers

## Q1. What is the difference between MCAR, MAR, and MNAR?

**Answer:** MCAR has no systematic cause related to the data. MAR can be explained using observed variables. MNAR depends on the missing value itself or an unobserved cause related to it.

## Q2. When is median imputation usually safer than mean imputation?

**Answer:** When a numeric feature is skewed or contains strong outliers, because the median is not pulled strongly by extreme values.

## Q3. Why can deleting every incomplete row be dangerous?

**Answer:** It can remove a large part of the dataset, reduce statistical power, and create bias if the removed rows are systematically different.

## Q4. Why is accuracy weak for a highly imbalanced problem?

**Answer:** A model can predict only the majority class and still obtain high accuracy while completely failing to find the minority class.

## Q5. What is the difference between random oversampling and SMOTE?

**Answer:** Random oversampling copies minority observations. SMOTE creates synthetic observations between nearby minority observations.

## Q6. Why must SMOTE be applied only after the train/test split?

**Answer:** Applying it before splitting lets related synthetic information appear across training and test data. This leakage makes evaluation look better than real performance.

## Q7. What does the 1.5 × IQR rule do?

**Answer:** It flags observations below `Q1 − 1.5×IQR` or above `Q3 + 1.5×IQR` as possible outliers. It is a flag, not an automatic deletion order.

## Q8. Why is label encoding risky for nominal input colours?

**Answer:** Integers create a fake order and fake distances. A model may treat red = 2 as greater than green = 1 even though colours have no ranking.

## Q9. When is ordinal encoding appropriate?

**Answer:** When categories have a real, defensible order, such as small < medium < large. You should explicitly provide that order.

## Q10. What is the main danger of target-guided encoding?

**Answer:** Target leakage. If a row's own target helps create its encoded feature, the model indirectly sees the answer. Use training-only mappings, out-of-fold encoding, smoothing, and a fallback for unseen categories.


# 12. Mini practice tasks

1. Add a missing-value indicator to `passengers["embarked"]`.
2. Compare the mean and median after adding age = 500.
3. Train a classifier on the imbalanced data with and without `class_weight="balanced"`.
4. Change the SMOTE interpolation value `r` from 0 to 1 and watch the synthetic point move.
5. Write a function that returns IQR outliers for any numeric `Series`.
6. One-hot encode `city`, `plan`, and a new unseen city.
7. Explain why `small=0`, `medium=1`, `large=2` is meaningful but `red=0`, `green=1`, `blue=2` is not.
8. Build a five-fold out-of-fold target encoder.
9. Replace the random split with a grouped or time split and explain why.
10. Compare precision, recall, F1, and balanced accuracy after changing the decision threshold.

## Final one-sentence summary

**Good feature engineering keeps useful signal, removes preventable mess, represents variables honestly, and protects evaluation from leakage.**


---

## Source coverage note

This notebook consolidates the supplied feature-engineering transcript and notebooks covering:

- missing values and MCAR/MAR/MNAR;
- row/column deletion and mean/median/mode/random imputation;
- imbalanced data, upsampling, and downsampling;
- SMOTE and its geometric intuition;
- five-number summaries, IQR fences, and box plots;
- one-hot, label, ordinal, and target-guided ordinal encoding.

The original demonstrations are retained in cleaner form. Errors that would make the code misleading or unsafe were corrected, and advanced sections add leakage prevention, pipelines, robust evaluation, unseen-category handling, smoothing, and deployment checks.
